In [3]:
def func_mrv(board):
    rows = [[False] * 10 for _ in range(9)]
    cols = [[False] * 10 for _ in range(9)]
    boxes = [[False] * 10 for _ in range(9)]
    empties = []

    def box_id(r, c):
        return (r // 3) * 3 + (c // 3)

    # Initialize constraints
    for r in range(9):
        for c in range(9):
            d = board[r][c]

            if d == 0:
                empties.append((r, c))
            else:
                b = box_id(r, c)
                rows[r][d] = True
                cols[c][d] = True
                boxes[b][d] = True

    def dfs(k):
        if k == len(empties):
            return True

        min_idx = k
        min_cnt = 10

        # Find the cell with the fewest candidates
        for i in range(k, len(empties)):
            r, c = empties[i]
            b = box_id(r, c)
            cnt = 0

            for d in range(1, 10):
                if not rows[r][d] and not cols[c][d] and not boxes[b][d]:
                    cnt += 1

            if cnt < min_cnt:
                min_cnt = cnt
                min_idx = i

            if min_cnt == 1:
                break

        if min_cnt == 0:
            return False

        # Move selected cell to position k
        empties[k], empties[min_idx] = empties[min_idx], empties[k]

        r, c = empties[k]
        b = box_id(r, c)

        for d in range(1, 10):
            if not rows[r][d] and not cols[c][d] and not boxes[b][d]:
                board[r][c] = d
                rows[r][d] = cols[c][d] = boxes[b][d] = True

                if dfs(k + 1):
                    return True

                # Backtrack
                board[r][c] = 0
                rows[r][d] = cols[c][d] = boxes[b][d] = False

        # Restore cell ordering
        empties[k], empties[min_idx] = empties[min_idx], empties[k]

        return False

    return dfs(0)

In [4]:

import pandas as pd
import time

df = pd.read_csv("../data/sudoku.csv")

def to_board(s):
    s = str(s)
    return [
        [int(s[i * 9 + j]) for j in range(9)]
        for i in range(9)
    ]

def benchmark(solver, df):
    correct = 0
    total_time = 0.0
    count = 0

    start_all = time.perf_counter()

    for row in df.itertuples(index=False):
        puzzle = row.quizzes
        expected = str(row.solutions)

        board = to_board(puzzle)

        start = time.perf_counter()
        result = solver(board)
        total_time += time.perf_counter() - start

        if result:
            actual = ''.join(
                str(cell)
                for line in board
                for cell in line
            )

            if actual == expected:
                correct += 1

        count += 1

        if count % 10000 == 0:
            print(f"Processed {count:,}/{len(df):,}")

    wall_time = time.perf_counter() - start_all

    return {
        "count": count,
        "correct": correct,
        "solver_time": total_time,
        "wall_time": wall_time,
        "average_ms": total_time * 1000 / count
    }


# print("Running old solver on the entire dataset...")
# old = benchmark(solve_sudoku, df)

print("\nRunning mrv solver on the entire dataset...")
new = benchmark(func_mrv, df)

# print("\n========== FINAL RESULTS ==========")
# print(f"Total puzzles: {old['count']:,}")

# print("\nOLD SOLVER")
# print(f"Correct: {old['correct']:,}/{old['count']:,}")
# print(f"Total solving time: {old['solver_time']:.3f} seconds")
# print(f"Average per puzzle: {old['average_ms']:.6f} ms")
# print(f"Total elapsed time: {old['wall_time']:.3f} seconds")

print("\nMRV SOLVER")
print(f"Correct: {new['correct']:,}/{new['count']:,}")
print(f"Total solving time: {new['solver_time']:.3f} seconds")
print(f"Average per puzzle: {new['average_ms']:.6f} ms")
print(f"Total elapsed time: {new['wall_time']:.3f} seconds")



Running mrv solver on the entire dataset...
Processed 10,000/1,000,000
Processed 20,000/1,000,000
Processed 30,000/1,000,000
Processed 40,000/1,000,000
Processed 50,000/1,000,000
Processed 60,000/1,000,000
Processed 70,000/1,000,000
Processed 80,000/1,000,000
Processed 90,000/1,000,000
Processed 100,000/1,000,000
Processed 110,000/1,000,000
Processed 120,000/1,000,000
Processed 130,000/1,000,000
Processed 140,000/1,000,000
Processed 150,000/1,000,000
Processed 160,000/1,000,000
Processed 170,000/1,000,000
Processed 180,000/1,000,000
Processed 190,000/1,000,000
Processed 200,000/1,000,000
Processed 210,000/1,000,000
Processed 220,000/1,000,000
Processed 230,000/1,000,000
Processed 240,000/1,000,000
Processed 250,000/1,000,000
Processed 260,000/1,000,000
Processed 270,000/1,000,000
Processed 280,000/1,000,000
Processed 290,000/1,000,000
Processed 300,000/1,000,000
Processed 310,000/1,000,000
Processed 320,000/1,000,000
Processed 330,000/1,000,000
Processed 340,000/1,000,000
Processed 35